# Fetch — MCTS-vs-softFloyd ablation (E1a + E1c)

Chạy trên **Kaggle GPU T4** (env MuJoCo cần stack cũ). Đo robustness của planner khi
world-model bị nhiễu: **E1a** (stochastic) + **E1c** (bias cố định + execution feedback).

**Trước khi chạy:** Settings → Accelerator = **GPU T4**, Internet = **ON**;
**Add Data →** output notebook đã train `fetch_s967` (để restore checkpoint). Eval chỉ cần
`agent.pt`+`algo.pt` (KHÔNG cần replay). Fetch = short-horizon (control): planning ít đòn bẩy, dự kiến hoà.

## 1. Code + env (~10–15 phút lần đầu)

In [ ]:
import os
if os.path.isdir('/kaggle/working/latent_landmarks'):
    !cd /kaggle/working/latent_landmarks && git pull -q origin retrain
else:
    !git clone -q -b retrain https://github.com/Jun1801/latent_landmarks.git /kaggle/working/latent_landmarks
if not os.path.isdir('/kaggle/working/wmag'):
    !git clone -q https://github.com/LunjunZhang/world-model-as-a-graph /kaggle/working/wmag
!bash /kaggle/working/latent_landmarks/repro/setup_kaggle.sh

## 2. Restore checkpoint (chỉ agent.pt + algo.pt)
Tìm `<SLUG>` bằng `!ls /kaggle/input/` (là dataset/output bạn vừa Add Data).

In [ ]:
!ls /kaggle/input/
import os, shutil
SLUG = 'PUT-DATASET-SLUG-HERE'          # <-- sửa cho khớp /kaggle/input/
CKPT, ENV = 'fetch_s967', 'FetchPickAndPlace-v1'
src = f'/kaggle/input/{SLUG}/experiments/{ENV}/{CKPT}/state'
dst = f'/kaggle/working/experiments/{ENV}/{CKPT}/state'; os.makedirs(dst, exist_ok=True)
for f in ['agent.pt', 'algo.pt']:
    shutil.copy(f'{src}/{f}', f'{dst}/{f}')
print('restored ->', os.listdir(dst))

## 3. Verify GPU + env

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python -c "import torch,mujoco_py; print('cuda',torch.cuda.is_available())"

## 4. E1a — σ=0 sanity (mcts ≈ soft_floyd ≈ checkpoint success)

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env fetch --regime e1a --resume_ckpt fetch_s967 --episodes 2 --n_test_rollouts 20 --sims 100 --sigmas 0

## 5. E1a — sweep (MCTS có robust hơn soft_floyd dưới nhiễu stochastic?)

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env fetch --regime e1a --resume_ckpt fetch_s967 --episodes 3 --n_test_rollouts 30 --sims 100 --sigmas 0 5 10 20 40

## 6. E1c — σ=0 sanity (soft_floyd/mcts_nofb/mcts_fb trùng nhau)

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env fetch --regime e1c --resume_ckpt fetch_s967 --episodes 2 --n_test_rollouts 20 --sims 100 --sigmas 0

## 7. E1c — sweep (mcts_fb phát hiện+né bias, soft_floyd/mcts_nofb kẹt?)

In [ ]:
!export PATH=/opt/conda/bin:$PATH; export LD_LIBRARY_PATH=$HOME/.mujoco/mujoco200/bin:/usr/lib/nvidia:${LD_LIBRARY_PATH:-};  conda run -n l3p python /kaggle/working/latent_landmarks/repro/paper_mcts/eval_ablation.py --env fetch --regime e1c --resume_ckpt fetch_s967 --episodes 3 --n_test_rollouts 30 --sims 100 --sigmas 0 0.1 0.3

## Ghi chú
- **σ=0 luôn phải khớp** soft_floyd(clean) — nếu lệch nhiều là port sai, dừng & báo.
- Kết quả in ra bảng success theo σ; copy lại để tổng hợp.
- Chạy dài (nhiều σ × episode × MCTS search) có thể vài chục phút — giảm `--episodes`/`--sims`
  nếu muốn nhanh; `--no_cuda` nếu không có GPU (chậm hơn nhiều).